# `predict`

Inference helpers used by the Streamlit app (and usable from notebooks / APIs).

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import json

import joblib
import numpy as np
import pandas as pd

from src import config as C

In [ ]:
def load_yield_artifacts():
    return joblib.load(C.MODELS / "yield_best.joblib"), json.loads((C.MODELS / "yield_meta.json").read_text())

In [ ]:
def load_reco_artifacts():
    return joblib.load(C.MODELS / "recommend_best.joblib")

In [ ]:
def predict_yield(model, meta, state, district, crop, season, year, area, rainfall_mm=None):
    """Returns predicted yield (production per unit area) and implied production."""
    row = {"State": state, "District": district, "Crop": crop, "Season": season,
           "Year": year, "log_area": float(np.log1p(area))}
    if meta["has_rain"]:
        row["rainfall_mm"] = rainfall_mm if rainfall_mm is not None else meta["state_rain"].get(state)
    X = pd.DataFrame([row])[meta["features"]]
    y = float(np.expm1(max(model.predict(X)[0], 0)))
    return y, y * area

In [ ]:
def recommend_crops(bundle, N, P, K, temperature, humidity, ph, rainfall, top_k=3):
    X = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=bundle["features"])
    proba = bundle["model"].predict_proba(X)[0]
    idx = np.argsort(proba)[::-1][:top_k]
    return [(bundle["classes"][i], float(proba[i])) for i in idx]